# Sommelier — Vietnamese UV Isolated Venv Test on Kaggle (2x T4 GPU)

Runs the sommelier podcast pipeline inside a **clean isolated `venv`** (`/kaggle/working/sommelier_env`) using **`uv`** on Kaggle with dual T4 GPUs.
This notebook dynamically patches runtime dependencies and handles Kaggle paths without modifying repository source files.


## 1. Sanity check the Kaggle runtime & GPUs

In [46]:
!nvidia-smi
!python --version
!df -h /kaggle/working 2>/dev/null || df -h .
import torch
print('PyTorch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Device count (GPUs):', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))


Sat Aug 15 03:40:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Setup Working Directory & Repository

In [47]:
import os
import shutil

BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
PROJECT_DIR = os.path.join(BASE_DIR, 'sommerlier')
ENV_DIR = os.path.join(BASE_DIR, 'sommelier_env')
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')

print(f"BASE_DIR: {BASE_DIR}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ENV_DIR: {ENV_DIR}")
print(f"AUDIO_DIR: {AUDIO_DIR}")

# Luôn xóa repo cũ và clone lại
if os.path.exists(PROJECT_DIR):
    print(f"Removing old repository: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

os.chdir(BASE_DIR)

print("Cloning repository...")
!git clone https://github.com/foresst123/sommerlier.git

print("Clone completed.")

BASE_DIR: /kaggle/working
PROJECT_DIR: /kaggle/working/sommerlier
ENV_DIR: /kaggle/working/sommelier_env
AUDIO_DIR: /kaggle/working/vi_audio
Removing old repository: /kaggle/working/sommerlier
Cloning repository...
Cloning into 'sommerlier'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 143 (delta 57), reused 131 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 22.33 MiB | 15.38 MiB/s, done.
Resolving deltas: 100% (57/57), done.
Clone completed.


## 3. Install dependencies into isolated venv via UV (~10 min)

Mirrors the three-step install order (torch CUDA 12.6 first, then requirements).


In [60]:
# 1. Install uv package manager
!pip install -q uv yt-dlp

# 2. Create clean isolated virtual environment
!uv venv --allow-existing {ENV_DIR}

# 3. Define proposed optimized dependencies list
PROPOSED_REQUIREMENTS = """
numpy==2.2.2
torch==2.8.0
torchaudio==2.8.0
torchvision==0.23.0
lightning==2.4.0
torchmetrics==1.6.2
onnxruntime-gpu>=1.20.0
nemo-toolkit[asr]==2.1.0
pyannote.audio==4.0.7
speechbrain==1.0.2
faster-whisper==1.2.0
whisperx==3.8.6
ctranslate2==4.5.0
demucs>=4.0.1
panns-inference
librosa==0.10.2.post1
soundfile==0.13.1
pydub==0.25.1
julius==0.2.7
numba==0.61.2
transformers==5.13.0
huggingface-hub>=1.5.0,<2.0
g2pk
jamo
nltk==3.9.1
openai==1.63.0
tritony==0.0.20
tritonclient[all]
pandas==2.2.3
PyYAML==6.0.2
tqdm==4.67.1
wandb==0.19.6
requests==2.32.4
einops==0.8.1
hydra-core==1.3.2
omegaconf==2.3.0
yt-dlp
setuptools>=75.0.0
sacrebleu

"""

req_file = os.path.join(BASE_DIR, 'requirements_proposed.txt')
with open(req_file, 'w') as f:
    f.write(PROPOSED_REQUIREMENTS.strip())

print('>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...')
!uv pip install --python {ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

print('\n>>> Step 2: Installing proposed dependencies into venv via UV...')
!uv pip install --python {ENV_DIR} -r {req_file} --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: /kaggle/working/sommelier_env
Activate with: source /kaggle/working/sommelier_env/bin/activate
>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...
Using Python 3.12.13 environment at: /kaggle/working/sommelier_env
Checked 3 packages in 6ms

>>> Step 2: Installing proposed dependencies into venv via UV...
Using Python 3.12.13 environment at: /kaggle/working/sommelier_env
  × No solution found when resolving dependencies:                                  
  ╰─▶ Because whisperx==3.8.6 depends on huggingface-hub<1.0.0 and you require
      whisperx==3.8.6, we can conclude that you require huggingface-hub<1.0.0.
      And because you require huggingface-hub>=1.5.0,<2.0, we can conclude
      that your requirements are unsatisfiable.


## 4. Hugging Face authentication & Config update

In [49]:
# Authenticate HuggingFace Token (supports Kaggle Secrets & Interactive Input)
import json, pathlib

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Fetched HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print("Kaggle Secrets not available or HF_TOKEN secret missing.")

if not hf_token:
    from getpass import getpass
    hf_token = getpass('Enter HF Token (hf_...): ')

from huggingface_hub import login
login(token=hf_token)

# Update config.json in podcast-pipeline
cfg_path = os.path.join(PROJECT_DIR, 'podcast-pipeline', 'config.json')
if os.path.exists(cfg_path):
    cfg = json.loads(pathlib.Path(cfg_path).read_text())
    cfg['huggingface_token'] = hf_token
    pathlib.Path(cfg_path).write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
    print(f'Updated Hugging Face token in {cfg_path}')


Fetched HF_TOKEN from Kaggle Secrets.
Updated Hugging Face token in /kaggle/working/sommerlier/podcast-pipeline/config.json


## 5. Prepare Audio Input (Auto-detects Kaggle Dataset `/kaggle/input`, Drag-Drop, or Fallback)

In [50]:
import os, glob, random, shutil, pathlib
import torch, torchaudio

# Always prepare a writable working audio folder
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

# 1. Auto-detect if user attached audio via Kaggle Datasets (/kaggle/input/...)
input_audio_files = []
if os.path.exists('/kaggle/input'):
    extensions = ('*.mp3', '*.wav', '*.flac', '*.m4a', '*.aac', '*.ogg')
    for ext in extensions:
        input_audio_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if input_audio_files:
    print(f"✅ Detected {len(input_audio_files)} audio file(s) in /kaggle/input/:")
    for f in input_audio_files:
        print("  -", f)
        dst = os.path.join(AUDIO_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst)
    print(f"Sync completed into writable folder: {AUDIO_DIR}")

# 2. Scan audio files in /kaggle/working/vi_audio/
audio_extensions = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')
audio_files = [f for f in glob.glob(os.path.join(AUDIO_DIR, '*')) if f.lower().endswith(audio_extensions)]

# 3. Fallback: If no audio files exist, generate a 10s synthetic test WAV file
if not audio_files:
    print('\n⚠️ No audio files found in /kaggle/input or /kaggle/working/vi_audio/.')
    print('Generating synthetic benchmark WAV file in vi_audio for smoke testing...')
    sample_rate = 16000
    duration_sec = 10
    t = torch.linspace(0, duration_sec, sample_rate * duration_sec)
    waveform = 0.3 * torch.sin(2 * 3.14159 * 440 * t) + 0.2 * torch.sin(2 * 3.14159 * 880 * t)
    waveform = waveform.unsqueeze(0)
    
    fallback_wav = os.path.join(AUDIO_DIR, 'kaggle_test_sample.wav')
    torchaudio.save(fallback_wav, waveform, sample_rate)
    audio_files = [fallback_wav]
    print(f'✅ Fallback test WAV created at: {fallback_wav}')

print(f'\n✅ Total audio file(s) ready for pipeline: {len(audio_files)}')
for af in audio_files:
    print('  -', af)


✅ Detected 1 audio file(s) in /kaggle/input/:
  - /kaggle/input/datasets/kieuduclamk18hl/data-thu-that-thach/thu_that_thach_10m.mp3
Sync completed into writable folder: /kaggle/working/vi_audio

✅ Total audio file(s) ready for pipeline: 1
  - /kaggle/working/vi_audio/thu_that_thach_10m.mp3


## 6. Dynamic In-Notebook Patches & Run Pipeline on 2x T4 GPUs

In [51]:
# Dynamically apply in-notebook patches without altering original repo files outside this execution
import glob, os

# Patch 1: Dynamically patch pkg_resources in venv site-packages regardless of Python version
site_pkgs = glob.glob(os.path.join(ENV_DIR, 'lib', 'python*', 'site-packages'))
if site_pkgs:
    pkg_res_file = os.path.join(site_pkgs[0], 'pkg_resources.py')
    with open(pkg_res_file, 'w', encoding='utf-8') as f:
        f.write('def declare_namespace(name): pass\n')
    print(f'✅ Dynamic patch applied to {pkg_res_file}')

# Patch 2: Dynamically patch main_original_ASR_MoE.py for PyTorch 2.x weights_only compatibility
main_script = os.path.join(PROJECT_DIR, 'podcast-pipeline', 'main_original_ASR_MoE.py')
if os.path.exists(main_script):
    with open(main_script, 'r', encoding='utf-8') as f:
        code = f.read()
    
    target_old = 'def _patched_load(path_or_url: Union[IO, str, Path], map_location=None) -> Any:'
    target_new = 'def _patched_load(path_or_url: Union[IO, str, Path], map_location=None, weights_only=False, **kwargs) -> Any:'
    
    if target_old in code:
        code = code.replace(target_old, target_new)
        with open(main_script, 'w', encoding='utf-8') as f:
            f.write(code)
        print(f'✅ Dynamic patch applied to {main_script}')
    else:
        print(f'ℹ️ {main_script} already patched or line signature differs.')
    
    target_import = "from chunkformer import ChunkFormerModel"
    if target_import in code:
        code = code.replace(target_import, f"# {target_import}")
        with open(main_script, "w", encoding="utf-8") as f:
            f.write(code)
        print(f"✅ Dynamic patch applied to remove chunkformer import in {main_script}")
    
# Patch 6: Fix WhisperX 3.8.6 missing whisperx.types module
whisper_asr_script = os.path.join(PROJECT_DIR, "podcast-pipeline", "models", "whisper_asr.py")
if os.path.exists(whisper_asr_script):
    with open(whisper_asr_script, "r", encoding="utf-8") as f:
        wasr_code = f.read()
    old_imp = "from whisperx.types import TranscriptionResult, SingleSegment"
    if old_imp in wasr_code and "try:" not in wasr_code.split(old_imp)[0][-10:]:
        new_imp = "try:\\n    from whisperx.types import TranscriptionResult, SingleSegment\\nexcept (ImportError, ModuleNotFoundError):\\n    TranscriptionResult = dict\\n    SingleSegment = dict"
        wasr_code = wasr_code.replace(old_imp, new_imp)
        with open(whisper_asr_script, "w", encoding="utf-8") as f:
            f.write(wasr_code)
        print(f"✅ Patch 6: fixed whisperx.types import in {whisper_asr_script}")
    
# Patch 7: Fix pyannote 4.x API change use_auth_token -> token
if os.path.exists(main_script):
    with open(main_script, "r", encoding="utf-8") as f:
        code = f.read()
    if "use_auth_token=" in code:
        code = code.replace("use_auth_token=", "token=")
        with open(main_script, "w", encoding="utf-8") as f:
            f.write(code)
        print(f"✅ Patch 7: use_auth_token -> token in {main_script}")
    
# Patch 8: Dual-GPU model allocation
if os.path.exists(main_script):
    with open(main_script, 'r', encoding='utf-8') as f:
        code = f.read()
    changed = False
    # 8a: device setup
    old_dev = 'device_name = "cuda"\n        device = torch.device(device_name)'
    new_dev = 'n_gpus = torch.cuda.device_count()\n        device_name = "cuda:0"\n        device = torch.device(device_name)\n        if n_gpus >= 2:\n            device_2 = torch.device("cuda:1")\n            logger.info(f"Dual-GPU mode: GPU 0 + GPU 1")\n        else:\n            device_2 = device\n            logger.info("Single-GPU mode")'
    if old_dev in code:
        code = code.replace(old_dev, new_dev)
        # Also add device_2 = device in CPU branch
        code = code.replace('device = torch.device(device_name)\n        # whisperX', 'device = torch.device(device_name)\n        device_2 = device\n        # whisperX')
        changed = True
    # 8b: PhoWhisper -> device_2
    if "device=device," in code and "PhoWhisper" in code:
        # Only replace the one inside the VN MoE block
        code = code.replace("model=\"vinai/PhoWhisper-large\",\n                device=device,", "model=\"vinai/PhoWhisper-large\",\n                device=device_2,")
        changed = True
    # 8c: Qwen3-ASR -> device_2
    if 'device_map="auto"' in code:
        code = code.replace('device_map="auto"', 'device_map={"": device_2}')
        changed = True
    # 8d: Sortformer -> device_2
    old_sort = "diar_model.eval()"
    new_sort = "diar_model = diar_model.to(device_2)\n    diar_model.eval()"
    if old_sort in code and "diar_model.to(device_2)" not in code:
        code = code.replace(old_sort, new_sort, 1)
        changed = True
    if changed:
        with open(main_script, 'w', encoding='utf-8') as f:
            f.write(code)
        print(f"✅ Patch 8: Dual-GPU allocation applied to {main_script}")


✅ Dynamic patch applied to /kaggle/working/sommelier_env/lib/python3.12/site-packages/pkg_resources.py
✅ Dynamic patch applied to /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py
✅ Dynamic patch applied to remove chunkformer import in /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py
✅ Patch 8: Dual-GPU allocation applied to /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py


In [52]:
import sys
import subprocess

python_bin = sys.executable

code = """
import pyannote.audio
import inspect
from pyannote.audio.pipelines import SpeakerDiarization

print("pyannote.audio version:", pyannote.audio.__version__)
print("pyannote.audio path:", pyannote.audio.__file__)
print("SpeakerDiarization.__init__ signature:")
print(inspect.signature(SpeakerDiarization.__init__))
"""

result = subprocess.run(
    [python_bin, "-c", code],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)


STDERR:
Traceback (most recent call last):
  File "<string>", line 2, in <module>
ModuleNotFoundError: No module named 'pyannote'



In [53]:
import subprocess
main_script = "/kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py"
subprocess.run(["sed", "-i", "s/use_auth_token=/token=/g", main_script])
print("✅ Fixed: use_auth_token -> token")


✅ Fixed: use_auth_token -> token


In [54]:
import subprocess

python_bin = "/kaggle/working/sommelier_env/bin/python"

commands = [
    "import sys; print('Python:', sys.executable)",
    "import torch; print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('GPU:', torch.cuda.device_count())",
    "import ctranslate2; print('CT2:', ctranslate2.__version__); print('CT2 path:', ctranslate2.__file__); print('CT2 CUDA devices:', ctranslate2.get_cuda_device_count())",
    "import faster_whisper; print('faster-whisper:', faster_whisper.__version__)",
]

for code in commands:
    result = subprocess.run(
        [python_bin, "-c", code],
        capture_output=True,
        text=True
    )
    print(">", code)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

> import sys; print('Python:', sys.executable)
Python: /kaggle/working/sommelier_env/bin/python

> import torch; print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('GPU:', torch.cuda.device_count())
Torch: 2.7.1+cu126
Torch CUDA: 12.6
GPU: 2

> import ctranslate2; print('CT2:', ctranslate2.__version__); print('CT2 path:', ctranslate2.__file__); print('CT2 CUDA devices:', ctranslate2.get_cuda_device_count())
CT2: 4.5.0
CT2 path: /kaggle/working/sommelier_env/lib/python3.12/site-packages/ctranslate2/__init__.py
CT2 CUDA devices: 2

> import faster_whisper; print('faster-whisper:', faster_whisper.__version__)
faster-whisper: 1.2.0



In [55]:
# Run the pipeline configured for Kaggle 2x T4 GPU
os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))
python_bin = os.path.join(ENV_DIR, 'bin', 'python')
import sys
site_packages = os.path.join(ENV_DIR, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages')
nvidia_lib = f"{site_packages}/nvidia/cudnn/lib:{site_packages}/torch/lib"

import os
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + nvidia_lib

!CUDA_VISIBLE_DEVICES=0,1 {python_bin} main_original_ASR_MoE.py \
  --input_folder_path {AUDIO_DIR} \
  --lang vi \
  --vad \
  --dia3 \
  --ASRMoE \
  --no-demucs \
  --whisperx_word_timestamps \
  --no-qwen3omni \
  --no-sepreformer \
  --LLM case_0 \
  --seg_th 0.11 \
  --min_cluster_size 11 \
  --clust_th 0.5 \
  --merge_gap 2


/kaggle/working/sommelier_env/lib/python3.12/site-packages/pyannote/audio/core/io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.7.1+cu126) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 7: libavutil.s

## 7. Inspect Output

In [56]:
import glob, json, pathlib

result_folder = os.path.join(AUDIO_DIR, '_final')
json_paths = sorted(glob.glob(f'{result_folder}/**/*.json', recursive=True))
print(f'Found {len(json_paths)} result file(s):')
for p in json_paths:
    print(' -', p)

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    print('\nMetadata:')
    print(json.dumps(result.get('metadata', {}), indent=2, ensure_ascii=False))
    print(f"\nFirst 3 segments of {len(result.get('segments', []))}:")
    for seg in result.get('segments', [])[:3]:
        print(json.dumps(seg, indent=2, ensure_ascii=False))


Found 0 result file(s):


## Kaggle 2x T4 Troubleshooting & Tips

- **Dual GPU Allocation**: Specified `CUDA_VISIBLE_DEVICES=0,1` for multi-GPU runtime.
- **Kaggle Secrets**: Set `HF_TOKEN` in Kaggle Secrets (Add-ons -> Secrets) so Hugging Face models auto-authenticate.
- **Input Audio**: Supports Kaggle Datasets (`/kaggle/input/...`), direct drag-drop (`/kaggle/working/vi_audio/`), or synthetic benchmark fallback audio.
- **Isolated Venv**: `uv` isolates dependencies in `/kaggle/working/sommelier_env` to avoid Kaggle pre-installed package conflicts.


In [57]:
import pandas as pd

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    audio_name = result.get("metadata", {}).get("audio_name", pathlib.Path(json_paths[0]).stem)
    
    rows = []
    for seg in result.get("segments", []):
        # Format time
        start = seg.get("start", 0)
        end = seg.get("end", 0)
        time_str = f"{start:.2f} - {end:.2f}"
        
        # Get speaker
        speaker = seg.get("speaker", "Unknown")
        
        # Get text (fallback to whisper or ensemble if text is not available)
        text = seg.get("text") or seg.get("text_ensemble") or seg.get("text_whisper") or ""
        
        rows.append({
            "Speaker": speaker,
            "Time": time_str,
            "Audio": audio_name,
            "Text": text
        })
    
    df = pd.DataFrame(rows)
    # Configure pandas display to show full text
    pd.set_option("display.max_colwidth", None)
    display(df)
else:
    print("No results found to display.")


No results found to display.
